# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DipeshGhimire33/Flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content page/item**, identified by `content_id`.

The dataset contains performance measurements covering the **last 90 days**, along with comparisons between the **most recent 30 days and the previous 30 days**. These fields provide both an overall performance view and a more recent indication of performance change.

I will verify the observation structure by checking fields including:

* `impressions_90d`
* `clicks_90d`
* `sessions_90d`
* `impressions_last_30d`
* `clicks_last_30d`
* `sessions_last_30d`
* `impressions_prev_30d`
* `clicks_prev_30d`
* `sessions_prev_30d`

This structure is appropriate for the Refresh / Content Opportunity Scoring objective because it provides both **page-level performance signals** and **recent-versus-prior performance comparisons** that can help identify pages warranting human review.



In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df =pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: Feature / Label / Context / Excluded

Because the goal is to create an interpretable **Opportunity Score** for ranking pages for human review, I will use a small set of signals directly related to performance deterioration, freshness, search opportunity, and potential optimization opportunity.

### Features

| Feature                  | Role in scoring                                                                       |
| ------------------------ | ------------------------------------------------------------------------------------- |
| `impressions_last_30d`   | Measures recent search visibility.                                                    |
| `impressions_prev_30d`   | Provides the comparison baseline for recent search visibility.                        |
| `clicks_last_30d`        | Measures recent search traffic.                                                       |
| `clicks_prev_30d`        | Provides the comparison baseline for recent search traffic.                           |
| `search_volume`          | Represents potential search demand.                                                   |
| `impressions_90d`        | Indicates whether the page has meaningful existing search visibility.                 |
| `avg_position`           | Indicates the page's current search-ranking position. `0` will be treated as no data. |
| `ctr`                    | Helps identify pages receiving impressions but relatively few clicks.                 |
| `days_since_last_update` | Measures content freshness and potential staleness.                                   |

These features cover four main dimensions:

* **Performance deterioration:** recent 30-day performance versus the previous 30 days
* **Freshness:** time since the page was last updated
* **Search opportunity:** search demand and existing visibility
* **Optimization opportunity:** ranking position and CTR

### Label / Target

| Field                | Role                                                                                                                                                                                                                 |
| -------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `is_declining_label` | Existing descriptive label indicating whether a page is classified as declining. It will **not be used as an input feature**. If used for evaluation or comparison, it will remain separate from the scoring inputs. |

The Opportunity Score itself is **not an existing label**. It is the output we are constructing to rank pages for human review.

### Context

| Field          | Role                                                                                                                                                      |
| -------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `content_id`   | Identifies the page in the final ranked review queue. It is not used as a feature.                                                                        |
| `client_id`    | Used to identify/group pages by client and, if modeling is performed, for grouped train/test splitting. It is not used as a feature.                      |
| `content_type` | Contextual information that can be used to interpret or segment the resulting rankings, particularly because missingness patterns differ by content type. |
| `main_intent`  | Contextual information that can help interpret why a page received a particular score.                                                                    |

### Excluded

| Field                                                            | Why excluded                                                                                                                                              |
| ---------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `trend_direction`                                                | **Leakage:** it is derived from `trend_pct`, and the existing decline label is based on this trend information. It must not be used as a scoring feature. |
| `trend_pct`                                                      | **Leakage:** it is used to derive `trend_direction`, which underlies the decline label. It therefore should not be used as a scoring feature.             |
| `content_id` as a feature                                        | Pseudonymous identifier with no meaningful predictive/scoring information.                                                                                |
| `client_id` as a feature                                         | Pseudonymous identifier that could cause client-specific memorization rather than learning page characteristics.                                          |
| `impressions_90d` if redundant with another visibility component | May be removed if exploratory analysis shows that it adds little information beyond the recent performance measures.                                      |
| `sessions_90d`                                                   | Not necessary for the initial score because search performance can be represented more directly through impressions and clicks.                           |
| `sessions_last_30d`                                              | Not necessary for the initial score; it adds another traffic measure that may overlap with clicks.                                                        |
| `sessions_prev_30d`                                              | Not necessary for the initial score for the same reason.                                                                                                  |
| `pageviews_90d`                                                  | Not essential to the initial refresh-priority decision and may overlap with sessions/traffic measures.                                                    |
| `users_90d`                                                      | Not essential to the initial scoring objective.                                                                                                           |
| `engaged_sessions_90d`                                           | Useful for broader engagement analysis, but not necessary for the first version of the refresh score.                                                     |
| `ai_sessions_90d`                                                | Not directly necessary for identifying conventional content-refresh opportunities.                                                                        |
| `scroll_events_90d`                                              | Engagement signal, but not essential to the initial ranking.                                                                                              |
| `days_with_impressions`                                          | Potentially useful for deeper analysis, but not required for the baseline score.                                                                          |
| `days_with_sessions`                                             | Potentially useful for deeper analysis, but not required for the baseline score.                                                                          |
| `content_age_days`                                               | Overlaps with freshness information captured by `days_since_last_update`.                                                                                 |
| `age_tier`                                                       | Redundant with `content_age_days`/freshness information.                                                                                                  |
| `age_tier_order`                                                 | Redundant with the underlying age variables.                                                                                                              |
| `freshness_tier`                                                 | Redundant with `days_since_last_update`.                                                                                                                  |
| `word_count`                                                     | Not necessary for the first scoring version and should not be assumed to indicate refresh opportunity by itself.                                          |
| `word_count_tier`                                                | Redundant with `word_count`.                                                                                                                              |
| `char_count`                                                     | Not necessary for the initial score.                                                                                                                      |
| `char_count_tier`                                                | Redundant with `char_count`.                                                                                                                              |
| `competition`                                                    | Potentially useful for search-opportunity analysis, but not necessary for the initial baseline score.                                                     |
| `competition_level`                                              | Same reason; can be investigated later if competition materially improves prioritization.                                                                 |
| `cpc`                                                            | Potentially useful as a business-value signal, but not essential for identifying whether a page warrants refresh review.                                  |
| `impression_tier`                                                | Redundant with the underlying impression values.                                                                                                          |
| `position_tier`                                                  | Redundant with `avg_position`.                                                                                                                            |
| `provider_used`                                                  | Describes the data/content-production process rather than the page's refresh opportunity.                                                                 |
| `model_used`                                                     | Describes the production/tooling process rather than the page's refresh opportunity.                                                                      |

### Initial Feature Set

The initial Opportunity Score will therefore start with approximately **9 core variables**:

1. `impressions_last_30d`
2. `impressions_prev_30d`
3. `clicks_last_30d`
4. `clicks_prev_30d`
5. `search_volume`
6. `impressions_90d`
7. `avg_position`
8. `ctr`
9. `days_since_last_update`

This is intentionally small. The objective is not to maximize the number of variables, but to create a **transparent and defensible ranking system** that a content/SEO team can understand and act on.

Additional variables can be tested later to determine whether they provide meaningful incremental value rather than assuming that more features automatically produce a better score.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())
print("Unique clients:", df["client_id"].nunique())

Rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0
Unique clients: 32


In [21]:
trend_counts = df["trend_direction"].value_counts(dropna=False)
print(trend_counts)
print("Total:", trend_counts.sum())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
Total: 30000


This verifies both the individual counts and that they sum to the expected 30,000 pages.

In [16]:
window_cols = [ "impressions_90d", "clicks_90d", "sessions_90d", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d" ]
display(df[window_cols].notna().sum())

impressions_90d         30000
clicks_90d              30000
sessions_90d            30000
impressions_last_30d    30000
clicks_last_30d         30000
sessions_last_30d       30000
impressions_prev_30d    30000
clicks_prev_30d         30000
sessions_prev_30d       30000
dtype: int64

In [22]:
missing_columns = [c for c in window_cols if c not in df.columns]
print("Missing expected columns:", missing_columns)

Missing expected columns: []


In [23]:
missing_summary = ( df[window_cols] .isna() .agg(["sum", "mean"]) .T .rename(columns={"sum": "missing_count", "mean": "missing_rate"}) )
print(missing_summary)

                      missing_count  missing_rate
impressions_90d                 0.0           0.0
clicks_90d                      0.0           0.0
sessions_90d                    0.0           0.0
impressions_last_30d            0.0           0.0
clicks_last_30d                 0.0           0.0
sessions_last_30d               0.0           0.0
impressions_prev_30d            0.0           0.0
clicks_prev_30d                 0.0           0.0
sessions_prev_30d               0.0           0.0


In [25]:
df.groupby("content_type")["search_volume"].apply(
    lambda s: s.isna().mean()
).sort_values(ascending=False)


df.groupby("content_type")["word_count"].apply(
    lambda s: s.isna().mean()
).sort_values(ascending=False)

content_type
keyword article       0.282979
comparison article    0.000000
feedly article        0.000000
Name: word_count, dtype: float64

In [28]:
position_zero = (df["avg_position"] == 0).sum()
position_missing = df["avg_position"].isna().sum()
print("avg_position == 0:", position_zero)
print("avg_position is missing:", position_missing)

avg_position == 0: 1205
avg_position is missing: 0


In [29]:
rate_columns = [ "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct", ]
print(df[rate_columns].describe().T)

                   count       mean        std  min  25%   50%    75%    max
ctr              30000.0   0.510733   3.279162  0.0  0.0  0.07   0.29  100.0
engagement_rate  30000.0   2.534520   8.310096  0.0  0.0  0.00   1.35  100.0
scroll_rate      29875.0  18.212921  29.472768  0.0  0.0  5.00  23.53  300.0
ai_traffic_pct   30000.0   0.768196   7.429454  0.0  0.0  0.00   0.00  300.0


In [30]:
for col in rate_columns: print( col, "max =", df[col].max(), "values > 100 =", (df[col] > 100).sum() )

ctr max = 100.0 values > 100 = 0
engagement_rate max = 100.0 values > 100 = 0
scroll_rate max = 300.0 values > 100 = 119
ai_traffic_pct max = 300.0 values > 100 = 23


Rate verification: ctr and engagement_rate have maximum values of 100 and no observations above 100. scroll_rate and ai_traffic_pct can exceed 100, with maximum values of 300 and 119/23 observations above 100 respectively. These values will be retained as provided because the dataset definition specifies that their numerator and denominator originate from different measurement systems.

In [36]:
print(
    "Rows per client:",
    df.groupby("client_id").size().count()
)

Rows per client: 32


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **No causal proof:** The data cannot tell us whether refreshing a page will actually improve performance.
* **Unbalanced history:** Pages may have different amounts of historical data, so comparisons may not always be equally reliable.
* **GSC-only / limited early data:** Some early rows may have incomplete measurements because not all systems have the same history.
* **Overlapping windows:** The 30-day periods are part of the 90-day window, so they are not independent observations.
* **Missing business context:** The data does not capture factors such as strategic importance, content quality, revenue value, or editorial priorities.
* **Correlation ≠ causation:** Declining, old, or low-performing pages may be associated with refresh opportunities, but these signals do not prove what caused the performance.
* **Ranking is prioritization, not certainty:** The Opportunity Score identifies **pages worth human review**, not pages guaranteed to benefit from a refresh.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.